# RappiPlus: From Data to Business Decisions

**Introduction**

This project evaluates the performance of the **RappiPlus** service to support **data-driven business decisions**.

It works with several business datasets:

- **rappiplus_orders_raw.csv** → orders, prices, discounts and revenue
- **rappiplus_catalog.csv** → product costs, categories and suppliers
- **rappiplus_marketing_spend.csv** → marketing spend by channel and country
- **events / users / user_activity (SQL)** → user behavior on the platform
- **experiment_checkout_ui.csv** → results of an A/B test on the checkout

The analysis follows a clear, progressive logic:

1. Can we trust the data? (data quality in Python)
2. Is the business profitable? (revenue, costs and profit)
3. Where do users drop off? (conversion funnel)
4. Do users come back? (cohort retention)
5. Do changes have an impact? (statistical test)
6. Communicating the results (BI dashboard)

Throughout the project, data is turned into insights that answer key business questions and support **actionable recommendations**.

**Note on column names:** the source datasets use Spanish column names. They are kept as-is to match the original schema. Main translations:

| Column | Meaning | Column | Meaning |
|---|---|---|---|
| `id_pedido` | order id | `nombre_evento` | event name |
| `id_usuario` | user id | `id_sesion` | session id |
| `fecha_hora_pedido` | order timestamp | `fecha_registro` | signup date |
| `pais` | country | `dias_despues_registro` | days since signup |
| `dispositivo` | device | `activo` | active (1/0) |
| `fuente_referencia` | referral source | `tipo_plan` | plan type |
| `nombre_producto` | product name | `variante` | experiment variant (`control` / `tratamiento` = treatment) |
| `categoria_producto` | product category | `convirtio` | converted (1/0) |
| `cantidad` | quantity | `duracion_sesion` | session duration |
| `precio_unitario` | unit price | `costo_unitario` | unit cost |
| `monto_descuento` | discount amount | `proveedor` | supplier |
| `monto_total` | total amount | `gasto` | spend |
| `canal` | channel | `id_campaña` | campaign id |

The placeholder value `'No registrado'` means "not recorded".

## Step 1: Load and validate data quality

### 1.1 Data loading and quick look

Load the three business datasets (`orders`, `catalog` and `marketing`) and explore their structure before analyzing them.

In [55]:
# Import libraries
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest


In [56]:
# Load datasets
orders = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv')
catalog = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv')
marketing = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv')

**Initial Exploration and Validation Plan:**

Before modeling and visualization, the three main datasets (`orders`, `catalog` and `marketing`) are reviewed systematically in three phases:

1. **Structure and data types:** inspect dimensions, columns and nulls with `.info()` and quick looks (`.head()`).
2. **Statistical summary:** review ranges, distributions and key metrics with `.describe()`.
3. **Integrity and quality:** check for duplicate records, unique values (`.nunique()`) and logical anomalies (such as zeros in critical numeric variables) to ensure the analysis is reliable.

In [63]:
# Explore datasets
print('General information: orders dataset')
display(orders.info())
print()
print('Numeric columns, summary statistics')
display(orders.describe())
print()
print('Unique values')
print(orders.nunique())
print()

General information: orders dataset
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB


None


Numeric columns, summary statistics


,cantidad,precio_unitario,monto_descuento,monto_total
count,25050.000000,25050.000000,25050.000000,2.510000e+04
mean,7.092735,259.305549,4.500798,2.072680e+03
std,296.277003,138.726461,5.223010,9.894995e+04
min,-2.000000,20.030000,0.000000,-4.926500e+02
25%,1.000000,138.377500,0.000000,1.805075e+02
50%,2.000000,258.715000,0.000000,3.417500e+02
75%,2.000000,380.332500,10.000000,5.185800e+02
max,20000.000000,499.960000,15.000000,8.840200e+06



Unique values
id_pedido             25000
id_usuario             7642
fecha_hora_pedido       181
pais                      6
dispositivo               2
fuente_referencia         3
nombre_producto           7
categoria_producto        3
cantidad                  6
precio_unitario       19543
monto_descuento           4
monto_total           21536
dtype: int64



In [70]:
print('Dataset preview')
display(orders.head())
print()
print('Duplicate rows')
print(orders.duplicated().sum())
print()
print('Duplicate rows preview')
display(orders[orders.duplicated(keep=False)].sort_values(by=list(orders.columns)).head(6))
print()
print('Zeros in numeric columns')
print(orders[(orders['cantidad'] == 0) | (orders['precio_unitario'] == 0) | (orders['monto_total'] == 0)] )

Dataset preview


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28



Duplicate rows
100

Duplicate rows preview


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
10082,order_10082,user_690,2025-02-17,Argentina,desktop,social,Sneakers-Urban-42,Moda,2.0,221.34,0.0,442.67
25023,order_10082,user_690,2025-02-17,Argentina,desktop,social,Sneakers-Urban-42,Moda,2.0,221.34,0.0,442.67
10709,order_10709,user_6783,2025-02-16,Colombia,mobile,organic,Jacket-Winter-M,Moda,1.0,170.10,10.0,160.10
25037,order_10709,user_6783,2025-02-16,Colombia,mobile,organic,Jacket-Winter-M,Moda,1.0,170.10,10.0,160.10
10829,order_10829,user_7697,2025-01-23,Argentina,mobile,social,Phone-Pro-128GB,Electronica,2.0,115.54,0.0,231.08
25065,order_10829,user_7697,2025-01-23,Argentina,mobile,social,Phone-Pro-128GB,Electronica,2.0,115.54,0.0,231.08



Zeros in numeric columns
Empty DataFrame
Columns: [id_pedido, id_usuario, fecha_hora_pedido, pais, dispositivo, fuente_referencia, nombre_producto, categoria_producto, cantidad, precio_unitario, monto_descuento, monto_total]
Index: []


In [58]:
product_names = orders['nombre_producto'].value_counts()
print(product_names)

Vacuum-Pro-Black        4199
Blender-XL-Red          4195
Jacket-Winter-M         4192
Sneakers-Urban-42       4160
Laptop-Gaming-16GB      2794
Tablet-Standard-64GB    2780
Phone-Pro-128GB         2750
Name: nombre_producto, dtype: int64


In [66]:
print('General information: catalog dataset')
display(catalog.info())
print()
print('Numeric columns, summary statistics')
display(catalog.describe())
print()
print('Unique values')
print(catalog.nunique())

General information: catalog dataset
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 352.0+ bytes


None


Numeric columns, summary statistics


,costo_unitario
count,7.000000
mean,102.252857
std,111.011563
min,10.120000
25%,16.905000
50%,25.210000
75%,182.975000
max,280.680000



Unique values
nombre_producto       7
categoria_producto    3
costo_unitario        7
proveedor             7
dtype: int64


In [68]:
print('Duplicate rows')
display(catalog.duplicated().sum())
print()
print('Dataset preview')
display(catalog.head())

Duplicate rows


0


Dataset preview


,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"


In [60]:
print('General information: marketing dataset')
display(marketing.info())
print()
print('Numeric columns, summary statistics')
display(marketing.describe())
print()
print('Unique values')
print(marketing.nunique())

General information: marketing dataset
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB
None

Numeric columns, summary statistics
            gasto
count  1620.00000
mean   1772.74292
std     734.43294
min     501.11000
25%    1128.03000
50%    1782.42500
75%    2420.68500
max    2999.36000

Unique values
fecha          180
pais             3
id_campaña       9
canal            3
gasto         1612
dtype: int64

Duplicate rows
0

Dataset preview
        fecha      pais            id_campaña        canal    gasto
0  2025-01-01    Mexico        organic_Mexico      organic  2446.25
1  2025-01-01    

In [ ]:
print('Duplicate rows')
print(marketing.duplicated().sum())
print()
print('Dataset preview')
display(marketing.head())

### 1.2 Data quality review and cleaning

Detect and fix data issues that could affect the revenue, cost and profitability analysis. The three datasets are checked for:

- Date format validity (convert to the correct type)
- Invalid numeric values (no invalid negatives or zeros)
- Consistency of amounts
- Duplicate records
- Categorical variables

In [7]:
orders['fecha_hora_pedido']= pd.to_datetime(orders['fecha_hora_pedido'])
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           25100 non-null  object        
 1   id_usuario          25100 non-null  object        
 2   fecha_hora_pedido   25100 non-null  datetime64[ns]
 3   pais                24800 non-null  object        
 4   dispositivo         25080 non-null  object        
 5   fuente_referencia   25070 non-null  object        
 6   nombre_producto     25070 non-null  object        
 7   categoria_producto  25020 non-null  object        
 8   cantidad            25050 non-null  float64       
 9   precio_unitario     25050 non-null  float64       
 10  monto_descuento     25050 non-null  float64       
 11  monto_total         25100 non-null  float64       
dtypes: datetime64[ns](1), float64(4), object(7)
memory usage: 2.3+ MB


In [8]:
orders = orders.drop_duplicates().reset_index(drop=True)
orders.duplicated().sum()

0

**Duplicate rows treatment:**

Exact duplicate records (rows repeated in full) were identified and removed with `.drop_duplicates()`. This prevents overstating sales volume and revenue because of redundant records, ensures each order is counted only once and preserves the integrity of the database.

In [9]:
orders['pais'] = orders['pais'].astype(str).str.strip().str.title()
orders['pais'].unique()

array(['Argentina', 'Mexico', 'Colombia', 'Nan'], dtype=object)

In [10]:
categorical_columns = ['pais', 'dispositivo', 'fuente_referencia', 'nombre_producto', 'categoria_producto']
orders[categorical_columns] = orders[categorical_columns].fillna('No registrado')
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           25000 non-null  object        
 1   id_usuario          25000 non-null  object        
 2   fecha_hora_pedido   25000 non-null  datetime64[ns]
 3   pais                25000 non-null  object        
 4   dispositivo         25000 non-null  object        
 5   fuente_referencia   25000 non-null  object        
 6   nombre_producto     25000 non-null  object        
 7   categoria_producto  25000 non-null  object        
 8   cantidad            24950 non-null  float64       
 9   precio_unitario     24950 non-null  float64       
 10  monto_descuento     24950 non-null  float64       
 11  monto_total         25000 non-null  float64       
dtypes: datetime64[ns](1), float64(4), object(7)
memory usage: 2.3+ MB


**Categorical null values treatment:**

Empty records were found in qualitative variables (such as `dispositivo`, `fuente_referencia` or `nombre_producto`). Since missing labels do not prevent analyzing the numeric transaction metrics, and dropping those rows would mean an unnecessary loss of sales information, they were imputed with the categorical value `'No registrado'` ("not recorded"). This keeps all numeric records and makes it possible to later audit or group the volume of operations with no identified origin or channel.

In [11]:
orders[orders['cantidad'].isna()][['cantidad', 'precio_unitario', 'monto_descuento', 'monto_total']]

,cantidad,precio_unitario,monto_descuento,monto_total
74,NaN,NaN,NaN,595.85
75,NaN,NaN,NaN,458.15
76,NaN,NaN,NaN,319.75
77,NaN,NaN,NaN,227.55
78,NaN,NaN,NaN,432.39
79,NaN,NaN,NaN,527.15
80,NaN,NaN,NaN,711.18
81,NaN,NaN,NaN,41.08
82,NaN,NaN,NaN,431.55
83,NaN,NaN,NaN,279.35


**Numeric null values treatment:**

The rows with missing values in `cantidad` (quantity) also have NaN values in `precio_unitario` and `monto_descuento` at the same time. There is not enough information to recover the values from the general sales formula (`monto_total = cantidad × precio_unitario − descuento`), and imputing with zeros or measures of central tendency (mean/median) would alter the distribution and bias the business metrics. These records were therefore removed with `dropna(subset=['cantidad'])`. They represent a minimal share of the dataset, and this decision preserves the integrity of the analysis.

In [12]:
orders = orders.dropna(subset=['cantidad'])
orders.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 24950 entries, 0 to 24999
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           24950 non-null  object        
 1   id_usuario          24950 non-null  object        
 2   fecha_hora_pedido   24950 non-null  datetime64[ns]
 3   pais                24950 non-null  object        
 4   dispositivo         24950 non-null  object        
 5   fuente_referencia   24950 non-null  object        
 6   nombre_producto     24950 non-null  object        
 7   categoria_producto  24950 non-null  object        
 8   cantidad            24950 non-null  float64       
 9   precio_unitario     24950 non-null  float64       
 10  monto_descuento     24950 non-null  float64       
 11  monto_total         24950 non-null  float64       
dtypes: datetime64[ns](1), float64(4), object(7)
memory usage: 2.5+ MB


In [13]:
orders[orders['cantidad'] < 0]

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
266,order_266,user_7011,2025-03-13,Nan,desktop,paid_search,Phone-Pro-128GB,Electronica,-2.0,101.31,10.0,-192.62
267,order_267,user_1087,2025-05-07,Nan,desktop,social,Phone-Pro-128GB,Electronica,-1.0,43.50,5.0,-38.50
268,order_268,user_84,2025-02-19,Nan,desktop,organic,Phone-Pro-128GB,Electronica,-1.0,497.65,5.0,-492.65
269,order_269,user_3323,2025-05-25,Nan,desktop,paid_search,Phone-Pro-128GB,Electronica,-1.0,423.53,0.0,-423.53


**Negative values:**

The observations with negative values in `cantidad` and `monto_total` keep strict mathematical consistency (`monto_total = cantidad × precio_unitario − descuento`) with positive unit prices. This behavior reflects operational transactions such as returns or refunds. Keeping these records is essential to accurately reflect the company's real net revenue and to avoid overstating total sales.

In [14]:
orders['cantidad'] = orders['cantidad'].astype(int)

In [15]:
marketing[marketing['canal'].isna()]

,fecha,pais,id_campaña,canal,gasto
98,2025-01-11,Argentina,social_Argentina,NaN,849.70
99,2025-01-12,Mexico,organic_Mexico,NaN,2033.56
100,2025-01-12,Mexico,paid_search_Mexico,NaN,1260.65
101,2025-01-12,Mexico,social_Mexico,NaN,1660.90
102,2025-01-12,Colombia,organic_Colombia,NaN,1819.27
...,...,...,...,...,...
194,2025-01-22,Colombia,social_Colombia,NaN,772.46
195,2025-01-22,Argentina,organic_Argentina,NaN,1562.61
196,2025-01-22,Argentina,paid_search_Argentina,NaN,644.34
197,2025-01-22,Argentina,social_Argentina,NaN,2544.66


In [16]:
marketing['canal'] = marketing['id_campaña'].str.rsplit('_', n=1).str[0]
marketing['canal'].unique()

array(['organic', 'paid_search', 'social'], dtype=object)

**Missing values in `canal` (~6%):**

The channel label was recovered directly from the campaign identifier (`id_campaña`). Using `rsplit('_', n=1)`, the country suffix at the end of each string was stripped, which preserves compound channel names intact (e.g. `paid_search`). This logical imputation recovers 100% of the attribution of advertising spend without distorting the marketing metrics.

In [17]:
marketing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1620 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB


**Note:** general amount ranges were validated at this stage. The critical finding, the isolation of the 10 outlier orders, is detailed in Step 2 (business analysis) because of its high impact on the metrics.

**Export:** once cleaning is finished, the datasets are exported to be used in the last stage of the project (the Tableau dashboard).

In [18]:
# Export cleaned datasets
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

## Step 2: Is the business profitable?

### 2.1 Main KPIs

Calculate the key indicators to assess revenue, costs and profitability, using the three datasets (`orders`, `catalog` and `marketing`).

**Financial and Operational Assessment Plan:**

This section joins sales transactions with the cost catalog and advertising spend to answer the key business questions:

1. **Overall financial health:** revenue, COGS, marketing investment and preliminary net profit.
2. **Behavioral metrics:** average ticket, units per order, best-selling product and spend by channel.
3. **Anomaly audit:** identification and isolation of outlier transactions to obtain the real, organic metrics of the operation.

In [19]:
merged_orders_catalog = pd.merge(orders, catalog, on='nombre_producto', how='left')
merged_orders_catalog.head()

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto_x,cantidad,precio_unitario,monto_descuento,monto_total,categoria_producto_y,costo_unitario,proveedor
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2,332.69,0.0,665.37,Moda,189.31,Mcmillan-Rhodes
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1,176.86,5.0,171.86,Electrónica,25.21,Bowers LLC
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2,102.99,10.0,195.99,Hogar,176.64,Long-Reid
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1,257.87,15.0,242.87,Electrónica,25.21,Bowers LLC
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1,336.28,0.0,336.28,Hogar,176.64,Long-Reid


In [20]:
revenue = merged_orders_catalog['monto_total'].sum()
print(revenue)

51965834.26


In [21]:
cost = round((merged_orders_catalog['cantidad'] * merged_orders_catalog['costo_unitario']).sum(),2)
print(cost)

43124018.41


In [22]:
marketing_cost = marketing['gasto'].sum()
print(marketing_cost)

2871843.53


In [23]:
profit = round((revenue - cost - marketing_cost),2) 
print(profit)

5969972.32


After joining the orders with the catalog costs (`merged_orders_catalog`) and adding the aggregated advertising spend across channels, the net profit for the period was calculated. Subtracting total COGS and marketing investment from gross revenue, the business shows a positive profit of **$5,969,972.32**, confirming that the current operation is financially sustainable and profitable.

In [24]:
average_by_ticket = round((merged_orders_catalog['monto_total'].mean()),2)
print(average_by_ticket)

2082.8


In [25]:
average_by_quantity = round((merged_orders_catalog['cantidad'].mean()),2)
print(average_by_quantity)

7.12


In [26]:
print(merged_orders_catalog['cantidad'].max())

20000


In [27]:
orders_above_5 = (merged_orders_catalog['cantidad']>5).sum()
print (f"Orders with more than 5 items: {orders_above_5}")
print()
filtered_outliers = merged_orders_catalog[merged_orders_catalog['cantidad'] > 5]
print(filtered_outliers[['id_pedido', 'id_usuario', 'fecha_hora_pedido', 'nombre_producto', 'cantidad', 'precio_unitario', 'costo_unitario', 'monto_descuento', 'monto_total']])

Orders with more than 5 items: 10

       id_pedido id_usuario fecha_hora_pedido     nombre_producto  cantidad  \
3471  order_3521  user_5812        2025-02-03  Laptop-Gaming-16GB     10000   
3472  order_3522  user_3575        2025-03-29  Laptop-Gaming-16GB     10000   
3536  order_3586  user_3380        2025-02-03  Laptop-Gaming-16GB     10000   
3593  order_3643  user_4440        2025-01-07  Laptop-Gaming-16GB     10000   
3606  order_3656   user_884        2025-01-01  Laptop-Gaming-16GB     20000   
3618  order_3668  user_7270        2025-06-24  Laptop-Gaming-16GB     20000   
3639  order_3689  user_6566        2025-06-16  Laptop-Gaming-16GB     10000   
3672  order_3722  user_4723        2025-05-09  Laptop-Gaming-16GB     20000   
3676  order_3726  user_2536        2025-02-12  Laptop-Gaming-16GB     20000   
3698  order_3748  user_7096        2025-02-23  Laptop-Gaming-16GB     10000   

      precio_unitario  costo_unitario  monto_descuento  monto_total  
3471            43.14    

In [28]:
median_by_quantity = merged_orders_catalog['cantidad'].median()
print(median_by_quantity)

2.0


In [29]:
median_by_ticket = merged_orders_catalog['monto_total'].median()
print(median_by_ticket)

341.56


In [30]:
clean_merged_orders_catalog = merged_orders_catalog[merged_orders_catalog['cantidad'] <= 50].copy()

#### Critical finding: anomaly detection and outlier filter

While evaluating sales behavior, a significant bias appeared when comparing the central tendency metrics for units per order and average ticket:

- **Mean** quantity per order: ~7.12 products
- **Median** quantity per order: 2.0 products
- **Maximum** detected: 20,000 products in a single transaction

**Diagnosis:** a deeper inspection of the dataset identified 10 anomalous transactions of the product `Laptop-Gaming-16GB` with extreme volumes (bulk purchases of 10,000 and 20,000 units) and severe inconsistencies in unit selling prices.

These 10 orders are extreme outliers that:

- Artificially inflate the mean average ticket and the mean quantity of products purchased.
- Distort the real consumption profile of the e-commerce's regular customers.

**Decision and data strategy:**

- **Global financials preserved:** the 10 orders are kept in the original dataset (`merged_orders_catalog`) to preserve the total accounting and billing evaluated in the profitability section (Part 1).
- **Analytical subset created (`clean_merged_orders_catalog`):** to answer customer behavior questions without distortion, a filtered dataset is generated that excludes transactions with more than 50 units.
- **Future audit:** these 10 orders are documented as system anomalies (possible platform stress tests or inventory loading errors) for review with the technical and operations teams.

In [31]:
avg_ticket_clean = round(clean_merged_orders_catalog['monto_total'].mean(), 2)
avg_quantity_clean = round(clean_merged_orders_catalog['cantidad'].mean(), 2)

print(f"Average ticket (without outliers): ${avg_ticket_clean}")
print(f"Average quantity (without outliers): {avg_quantity_clean} units")

Average ticket (without outliers): $385.77
Average quantity (without outliers): 1.5 units


In [32]:
clean_revenue = round(clean_merged_orders_catalog['monto_total'].sum(), 2)
clean_cost = round((clean_merged_orders_catalog['cantidad'] * clean_merged_orders_catalog['costo_unitario']).sum(), 2)
clean_profit = round(clean_revenue - clean_cost - marketing_cost, 2)

print("--- COMPARISON: BEFORE VS. AFTER REMOVING OUTLIERS ---")
print(f"Original Revenue:       ${revenue:,.2f}  -->  Clean: ${clean_revenue:,.2f}")
print(f"Original COGS:          ${cost:,.2f}   -->  Clean: ${clean_cost:,.2f}")
print(f"Original Profit:        ${profit:,.2f}   -->  Clean: ${clean_profit:,.2f}")

--- COMPARISON: BEFORE VS. AFTER REMOVING OUTLIERS ---
Original Revenue:       $51,965,834.26  -->  Clean: $9,621,134.26
Original COGS:          $43,124,018.41   -->  Clean: $3,828,818.41
Original Profit:        $5,969,972.32   -->  Clean: $2,920,472.32


After re-evaluating the accounting while isolating the 10 outlier orders (quantity > 50), a massive bias in the initial consolidated metrics becomes evident:

- **Organic revenue:** gross revenue from the normal customer operation falls from 51.96M to 9.62M, which reveals that 81.5% of the initially reported revenue belonged to only 10 anomalous transactions.
- **COGS adjustment:** cost of goods sold goes from 43.12M to 3.83M.
- **Real organic profit:** despite the drastic drop in theoretical revenue, the business remains in positive territory with a real net profit of **$2,920,472.32**.

**Conclusion:** although the raw accounting records showed sales above 51M, the commercial health and recurring behavior of the e-commerce operate at a scale close to 9.6M in revenue. Keeping these 10 transactions in the operational analysis would have led to completely wrong demand and inventory projections. For this reason, all user behavior, channel and funnel metrics in the following sections are evaluated exclusively on the `clean_merged_orders_catalog` subset.

In [33]:
clean_merged_orders_catalog['nombre_producto'].describe()

count              24940
unique                 8
top       Blender-XL-Red
freq                4176
Name: nombre_producto, dtype: object

**Best-selling product:** the item with the highest transaction volume in the analyzed period is `Blender-XL-Red`, with 4,176 successful transactions in the clean dataset.

**Catalog structure:** the official catalog has 7 main products. An eighth value, `No registrado` (30 records), was intentionally added during data cleaning to label originally null values without discarding valuable transaction records. This keeps overall sales volumes intact without distorting the core catalog.

In [34]:
marketing.groupby('canal')['gasto'].sum()

canal
organic        972650.96
paid_search    922374.20
social         976818.37
Name: gasto, dtype: float64

The marketing budget analysis shows a highly balanced and diversified distribution across the company's three acquisition channels, avoiding dependence on a single source.

### Closing note: interactive segmentation and data criteria

For a detailed, multidimensional exploration of the business, the **geographic and product catalog segmentation is available in the interactive Tableau dashboard** (linked at the end of this notebook).

**Cleaning criteria applied for the Tableau visualization:**

1. **Outlier treatment:** the 10 outlier orders of *Laptop-Gaming-16GB* (quantities of 10,000 and 20,000 units) were excluded so they would not skew chart scales, and to reflect the recurring behavior of users.
2. **Exclusion of `No registrado` records:** 30 records labeled *No registrado* (a marginal amount of about $12,000) were omitted. Although they were kept in the general accounting to preserve the integrity of total revenue, they were excluded from the graphical segmentation because they added no analytical value to the visual flow and their financial impact is irrelevant compared with the overall volume.

This way, the interactive dashboard offers a clean, representative experience focused on the real levers for decision-making.

## Step 3: Where are users lost? (conversion funnel)

Analyze user behavior to identify at which stage of the process users drop off. A connection to the database is established to run SQL queries on the `events` table.

In [35]:
import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

# Database credentials are read from environment variables (see .env.example)
load_dotenv()

db_config = {
    'user': os.getenv('DB_USER'),
    'pwd': os.getenv('DB_PWD'),
    'host': os.getenv('DB_HOST'),
    'port': int(os.getenv('DB_PORT', 5432)),
    'db': os.getenv('DB_NAME'),
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode': 'require'})


**Conversion Funnel Analysis Plan (SQL):**

This section connects directly to the database to evaluate friction and retention of users along the browsing funnel:

1. **Cumulative vs. sequential behavior:** comparison between independent interactions (`GROUP BY`) and the strict step-by-step user flow (chained `LEFT JOIN` starting from `first_visit`).
2. **Total (end-to-end) conversion:** percentage of users who complete the purchase, using the initial event as the base.
3. **Step-by-step conversion (friction points):** identification of specific drops between consecutive stages to isolate bottlenecks (especially at the payment gateway and when entering payment information).

In [36]:
# Explore the events table
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [37]:
# Funnel totals

query_totals = '''
SELECT 
    nombre_evento, 
    COUNT(DISTINCT id_usuario) AS total_users
FROM events
GROUP BY nombre_evento
ORDER BY 
    CASE nombre_evento
        WHEN 'first_visit' THEN 1
        WHEN 'select_item' THEN 2
        WHEN 'add_to_cart' THEN 3
        WHEN 'begin_checkout' THEN 4
        WHEN 'add_payment_info' THEN 5
        WHEN 'purchase' THEN 6
        ELSE 7
    END
'''

totals = pd.read_sql(query_totals, con=engine)
totals

,nombre_evento,total_users
0,first_visit,7796
1,select_item,7582
2,add_to_cart,7634
3,begin_checkout,7208
4,add_payment_info,6250
5,purchase,6240


**Interpretation:** this query counts the total number of unique users who performed each event independently during the analyzed period, without requiring a chronological or direct sequence within the same session.

**Finding:** it measures the overall demand for each interaction on the platform (for example, the total number of users who completed a purchase, regardless of whether they started browsing in this period or arrived through a direct checkout link).

In [38]:
# Funnel totals

query_totals_sequence = '''
WITH first_visit AS (
    SELECT 
        DISTINCT id_usuario
    FROM events
    where nombre_evento = 'first_visit'
),
select_item AS (
    SELECT 
        DISTINCT id_usuario
    FROM events
    where nombre_evento = 'select_item'
),
add_to_cart AS (
    SELECT 
        DISTINCT id_usuario
    FROM events
    where nombre_evento = 'add_to_cart'
),
begin_checkout AS (
    SELECT 
        DISTINCT id_usuario
    FROM events
    where nombre_evento = 'begin_checkout'
),
add_payment_info AS (
    SELECT 
        DISTINCT id_usuario
    FROM events
    where nombre_evento = 'add_payment_info'
),
purchase AS (
    SELECT 
        DISTINCT id_usuario
    FROM events
    where nombre_evento = 'purchase'
)
SELECT 
    COUNT (distinct a.id_usuario) AS users_first_visit,
    COUNT (distinct b.id_usuario) AS users_select_item,
    COUNT (distinct c.id_usuario) AS users_add_to_cart,
    COUNT (distinct d.id_usuario) AS users_begin_checkout,
    COUNT (distinct f.id_usuario) AS users_add_payment_info,
    COUNT (distinct g.id_usuario) AS users_purchase
FROM
    first_visit a
    LEFT JOIN select_item b on a.id_usuario = b.id_usuario
    LEFT JOIN add_to_cart c on b.id_usuario = c.id_usuario
    LEFT JOIN begin_checkout d on c.id_usuario = d.id_usuario
    LEFT JOIN add_payment_info f on d.id_usuario = f.id_usuario
    LEFT JOIN purchase g on f.id_usuario = g.id_usuario
'''

totals_sequence = pd.read_sql(query_totals_sequence, con=engine)
totals_sequence

,users_first_visit,users_select_item,users_add_to_cart,users_begin_checkout,users_add_payment_info,users_purchase
0,7796,7393,7052,6364,4967,3857


**Interpretation:** this query chains each stage of the flow sequentially with `LEFT JOIN`s, evaluating only the users who completed the journey step by step from their first visit (`first_visit → select_item → add_to_cart → begin_checkout → add_payment_info → purchase`).

**Finding:** it shows the strictly linear behavior. Of the 7,796 users who started the journey at `first_visit`, 3,857 completed all 6 stages of the funnel. The difference with the overall cumulative count reflects returning users, direct purchases (fast checkout) or skipped stages through external links.

In [39]:
# Funnel conversion rates

query_total_conversion = '''
WITH first_visit AS (
    SELECT DISTINCT id_usuario FROM events WHERE nombre_evento = 'first_visit'
),
select_item AS (
    SELECT DISTINCT id_usuario FROM events WHERE nombre_evento = 'select_item'
),
add_to_cart AS (
    SELECT DISTINCT id_usuario FROM events WHERE nombre_evento = 'add_to_cart'
),
begin_checkout AS (
    SELECT DISTINCT id_usuario FROM events WHERE nombre_evento = 'begin_checkout'
),
add_payment_info AS (
    SELECT DISTINCT id_usuario FROM events WHERE nombre_evento = 'add_payment_info'
),
purchase AS (
    SELECT DISTINCT id_usuario FROM events WHERE nombre_evento = 'purchase'
)
SELECT

    ROUND(COUNT(DISTINCT b.id_usuario) * 100.0 / NULLIF(COUNT(DISTINCT a.id_usuario), 0), 2) AS conv_select,
    ROUND(COUNT(DISTINCT c.id_usuario) * 100.0 / NULLIF(COUNT(DISTINCT a.id_usuario), 0), 2) AS conv_cart,
    ROUND(COUNT(DISTINCT d.id_usuario) * 100.0 / NULLIF(COUNT(DISTINCT a.id_usuario), 0), 2) AS conv_checkout,
    ROUND(COUNT(DISTINCT f.id_usuario) * 100.0 / NULLIF(COUNT(DISTINCT a.id_usuario), 0), 2) AS conv_payment,
    ROUND(COUNT(DISTINCT g.id_usuario) * 100.0 / NULLIF(COUNT(DISTINCT a.id_usuario), 0), 2) AS conv_purchase

FROM first_visit a
LEFT JOIN select_item b ON a.id_usuario = b.id_usuario
LEFT JOIN add_to_cart c ON b.id_usuario = c.id_usuario
LEFT JOIN begin_checkout d ON c.id_usuario = d.id_usuario
LEFT JOIN add_payment_info f ON d.id_usuario = f.id_usuario
LEFT JOIN purchase g ON f.id_usuario = g.id_usuario;
'''

conversion_total = pd.read_sql(query_total_conversion, con=engine)
conversion_total

,conv_select,conv_cart,conv_checkout,conv_payment,conv_purchase
0,94.83,90.46,81.63,63.71,49.47


**Interpretation:** this query calculates the cumulative retention at each funnel stage, using the total number of initial users (`first_visit`) as the absolute baseline.

**Finding:** it shows the overall health of the end-to-end conversion flow. 49.47% of the users who started their journey completed the purchase (`conv_purchase`), giving a clear and direct view of the funnel's global performance without local detours.

In [40]:
# Funnel conversion rates

query_vs_prev_conversion = '''
WITH first_visit AS (
    SELECT DISTINCT id_usuario FROM events WHERE nombre_evento = 'first_visit'
),
select_item AS (
    SELECT DISTINCT id_usuario FROM events WHERE nombre_evento = 'select_item'
),
add_to_cart AS (
    SELECT DISTINCT id_usuario FROM events WHERE nombre_evento = 'add_to_cart'
),
begin_checkout AS (
    SELECT DISTINCT id_usuario FROM events WHERE nombre_evento = 'begin_checkout'
),
add_payment_info AS (
    SELECT DISTINCT id_usuario FROM events WHERE nombre_evento = 'add_payment_info'
),
purchase AS (
    SELECT DISTINCT id_usuario FROM events WHERE nombre_evento = 'purchase'
)
SELECT

    ROUND(COUNT(DISTINCT b.id_usuario) * 100.0 / NULLIF(COUNT(DISTINCT a.id_usuario), 0), 2) AS select_vs_prev,
    ROUND(COUNT(DISTINCT c.id_usuario) * 100.0 / NULLIF(COUNT(DISTINCT b.id_usuario), 0), 2) AS add_cart_vs_prev,
    ROUND(COUNT(DISTINCT d.id_usuario) * 100.0 / NULLIF(COUNT(DISTINCT c.id_usuario), 0), 2) AS begin_checkout_vs_prev,
    ROUND(COUNT(DISTINCT f.id_usuario) * 100.0 / NULLIF(COUNT(DISTINCT d.id_usuario), 0), 2) AS add_payment_vs_prev,
    ROUND(COUNT(DISTINCT g.id_usuario) * 100.0 / NULLIF(COUNT(DISTINCT f.id_usuario), 0), 2) AS purchase_vs_prev

FROM first_visit a
LEFT JOIN select_item b ON a.id_usuario = b.id_usuario
LEFT JOIN add_to_cart c ON b.id_usuario = c.id_usuario
LEFT JOIN begin_checkout d ON c.id_usuario = d.id_usuario
LEFT JOIN add_payment_info f ON d.id_usuario = f.id_usuario
LEFT JOIN purchase g ON f.id_usuario = g.id_usuario;
'''

conversion_vs_prev = pd.read_sql(query_vs_prev_conversion, con=engine)
conversion_vs_prev

,select_vs_prev,add_cart_vs_prev,begin_checkout_vs_prev,add_payment_vs_prev,purchase_vs_prev
0,94.83,95.39,90.24,78.05,77.65


**Interpretation:** this query strictly evaluates retention between a step and the immediately preceding one (e.g. how many of those who reached `begin_checkout` moved on to `add_payment_info`).

**Finding:** it pinpoints where the greatest friction in the business occurs. The step from `begin_checkout` to `add_payment_info` shows a significant drop (only 78.05% of users are retained), which points to a critical optimization opportunity in the payment gateway or checkout form. Likewise, the step from `add_payment_info` to `purchase` retains only 77.65% of users, which is another point worth investigating.

## Step 4: Do users come back? (cohort retention)

Analyze user retention to understand whether users return after signing up. Tables used: `users` and `user_activity`.

**Methodology:**

1. Each user's **cohort** is defined by signup month. The signup date is cast with `CAST(fecha_registro AS DATE)`.
2. Weekly retention is calculated: how many users **remain active** in each week after signup.
   - `retained_w1`: users active in week 1
   - `retained_w2`: users active in week 2
   - `retained_w3`: users active in week 3
3. The retention percentage is calculated for each week by dividing retained users by the initial users of the cohort:
   - `week_1`: percentage of users retained in week 1
   - `week_2`: percentage of users retained in week 2
   - `week_3`: percentage of users retained in week 3

**Cohort Retention Analysis Plan:**

This section groups users by signup month (`cohort`) and analyzes their lifecycle and engagement over the following weeks using two complementary methodologies:

1. **Period-specific retention (`=`):** measures users active exactly on day 7, 14 and 21. It captures point-in-time, intermittent activity.
2. **Cumulative / rolling retention (`>=`):** measures users with sustained activity *at least* from that week onward. It reflects the true natural decay curve and retention of the platform.

In [41]:
# Explore the users table
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(5)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free
3,user_3,2025-03-04,Mexico,desktop,free
4,user_4,2025-02-27,Argentina,desktop,free


In [42]:
# Explore the user_activity table
query_users_activity = '''
SELECT *
FROM user_activity;
'''
users_activity = pd.read_sql(query_users_activity, con=engine)
users_activity.head(5)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1
3,user_0,2025-02-26,28,0
4,user_1,2025-01-14,7,0


In [43]:
# Cohort retention

query_cohort_retention_final = '''
WITH cohorts AS (
    SELECT 
        id_usuario,
        TO_CHAR(DATE_TRUNC('month', CAST(fecha_registro AS DATE)), 'YYYY-MM') AS cohort
    FROM users
),
activity AS (
    SELECT 
        c.id_usuario, 
        c.cohort,
        a.dias_despues_registro, 
        a.activo
    FROM cohorts c
    LEFT JOIN user_activity a ON c.id_usuario = a.id_usuario
)
SELECT 
    cohort,
    COUNT(DISTINCT CASE WHEN dias_despues_registro = 7 AND activo = 1 THEN id_usuario END) AS retained_w1,
    COUNT(DISTINCT CASE WHEN dias_despues_registro = 14 AND activo = 1 THEN id_usuario END) AS retained_w2,
    COUNT(DISTINCT CASE WHEN dias_despues_registro = 21 AND activo = 1 THEN id_usuario END) AS retained_w3,
    ROUND(COUNT(DISTINCT CASE WHEN dias_despues_registro = 7 AND activo = 1 THEN id_usuario END) * 100.0 / NULLIF(COUNT(DISTINCT id_usuario), 0), 2) AS week_1,
    ROUND(COUNT(DISTINCT CASE WHEN dias_despues_registro = 14 AND activo = 1 THEN id_usuario END) * 100.0 / NULLIF(COUNT(DISTINCT id_usuario), 0), 2) AS week_2,
    ROUND(COUNT(DISTINCT CASE WHEN dias_despues_registro = 21 AND activo = 1 THEN id_usuario END) * 100.0 / NULLIF(COUNT(DISTINCT id_usuario), 0), 2) AS week_3
FROM activity
WHERE cohort IS NOT NULL
GROUP BY cohort
ORDER BY cohort;

'''

# Run the query
cohort_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohort_final

,cohort,retained_w1,retained_w2,retained_w3,week_1,week_2,week_3
0,2025-01,697,668,656,42.84,41.06,40.32
1,2025-02,611,609,635,42.31,42.17,43.98
2,2025-03,677,705,690,41.38,43.09,42.18
3,2025-04,680,697,663,42.34,43.40,41.28
4,2025-05,695,676,706,41.20,40.07,41.85


**Conclusion: period-specific retention**

**Observed behavior:** retention rates remain stable at around 40% to 43%, independently in each evaluated week.

**Analytical interpretation:** by using exact conditions per week (`dias_despues_registro = X`), this approach shows that some users have intermittent activity (they are absent one week but reconnect in the following ones).

In [44]:

# Cohort retention

query_cohort_retention_final = '''
WITH cohorts AS (
    SELECT 
        id_usuario,
        TO_CHAR(DATE_TRUNC('month', CAST(fecha_registro AS DATE)), 'YYYY-MM') AS cohort
    FROM users
),
activity AS (

    SELECT 
        c.id_usuario, 
        c.cohort, 
        a.dias_despues_registro, 
        a.activo
    FROM cohorts c
    LEFT JOIN user_activity a ON c.id_usuario = a.id_usuario
)
SELECT 
    cohort,
    COUNT(DISTINCT CASE WHEN dias_despues_registro >= 7 AND activo = 1 THEN id_usuario END) AS retained_w1,
    COUNT(DISTINCT CASE WHEN dias_despues_registro >= 14 AND activo = 1 THEN id_usuario END) AS retained_w2,
    COUNT(DISTINCT CASE WHEN dias_despues_registro >= 21 AND activo = 1 THEN id_usuario END) AS retained_w3,
    ROUND(COUNT(DISTINCT CASE WHEN dias_despues_registro >= 7 AND activo = 1 THEN id_usuario END) * 100.0 / NULLIF(COUNT(DISTINCT id_usuario), 0), 2) AS week_1,
    ROUND(COUNT(DISTINCT CASE WHEN dias_despues_registro >= 14 AND activo = 1 THEN id_usuario END) * 100.0 / NULLIF(COUNT(DISTINCT id_usuario), 0), 2) AS week_2,
    ROUND(COUNT(DISTINCT CASE WHEN dias_despues_registro >= 21 AND activo = 1 THEN id_usuario END) * 100.0 / NULLIF(COUNT(DISTINCT id_usuario), 0), 2) AS week_3
FROM activity
WHERE cohort IS NOT NULL
GROUP BY cohort
ORDER BY cohort;

'''

# Run the query
cohort_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohort_final

,cohort,retained_w1,retained_w2,retained_w3,week_1,week_2,week_3
0,2025-01,1381,1253,1027,84.88,77.01,63.12
1,2025-02,1255,1154,940,86.91,79.92,65.10
2,2025-03,1428,1306,1060,87.29,79.83,64.79
3,2025-04,1394,1261,1022,86.80,78.52,63.64
4,2025-05,1446,1321,1088,85.71,78.30,64.49


**Conclusion: cumulative (rolling) retention**

**Observed behavior:** it shows the classic natural, downward decay curve of a cohort (starting at a strong ~85% in week 1 and progressively dropping to ~64% in week 3).

**Analytical interpretation:** the cumulative operator (`>=`) ensures a reading of sustained retention over time, showing accurately the volume of users who keep using the platform continuously since signing up.

## Step 5: Do changes have an impact? (statistical test)

**Goal:** evaluate whether the change to the checkout UI affects the **purchase conversion rate**.

The dataset `experiment_checkout_ui.csv` is analyzed around its main metric, `convirtio` (conversion): 1 if the user completed the purchase, 0 if not.

**Statistical Validation Plan (A/B Test):**

This section evaluates the impact of a change to the checkout user interface (*UI*) using statistical inference, to determine whether it improves the conversion rate:

1. **Experiment inspection:** review of the data structure, unique users and distribution of variants (*control* vs. *treatment*).
2. **Hypothesis statement:** formal definition of the null hypothesis ($H_0$) and the alternative hypothesis ($H_1$) with a significance level of $\alpha = 0.05$.
3. **Two-proportion Z-test:** run the statistical test to compare success rates between both groups.
4. **Conclusion and recommendation:** interpretation of the *p-value* to decide whether the change should be scaled to production, avoiding decisions based on random fluctuations.

**Statistical hypotheses**

- **H₀ (null hypothesis):** there is no significant difference in conversion rate between the control group and the experimental group (the UI change had no real impact).
- **H₁ (alternative hypothesis):** there is a significant difference in conversion rate between both groups (the UI change did have an impact).

**Statistical test:** Z-test for comparing proportions.

**Significance level (alpha):** the industry standard is 0.05 (5%), meaning we accept a 5% probability of being wrong when rejecting the null hypothesis.

In [45]:
experiment = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv')
experiment.head()

,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,tratamiento,0,mobile,Argentina,114.41,2025-03-28
1,exp_user_1,tratamiento,0,desktop,Mexico,170.03,2025-01-15
2,exp_user_2,control,1,mobile,Colombia,140.21,2025-03-18
3,exp_user_3,tratamiento,0,mobile,Colombia,151.45,2025-06-03
4,exp_user_4,tratamiento,0,desktop,Mexico,299.96,2025-01-12


In [46]:
experiment.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id_usuario       10000 non-null  object 
 1   variante         10000 non-null  object 
 2   convirtio        10000 non-null  int64  
 3   dispositivo      10000 non-null  object 
 4   pais             10000 non-null  object 
 5   duracion_sesion  10000 non-null  float64
 6   timestamp        10000 non-null  object 
dtypes: float64(1), int64(1), object(5)
memory usage: 547.0+ KB


In [47]:
experiment['timestamp'] = pd.to_datetime(experiment['timestamp'])
print(f"Number of unique users {experiment['id_usuario'].nunique()}")
print()
print('Date range')
print(experiment['timestamp'].describe())

Number of unique users 10000

Date range
count                   10000
unique                    181
top       2025-06-06 00:00:00
freq                       75
first     2025-01-01 00:00:00
last      2025-06-30 00:00:00
Name: timestamp, dtype: object


In [48]:
conversions = experiment.groupby('variante')['convirtio'].sum()
conversions

variante
control        779
tratamiento    820
Name: convirtio, dtype: int64

In [49]:
group_totals = experiment.groupby('variante')['convirtio'].count()
group_totals

variante
control        4965
tratamiento    5035
Name: convirtio, dtype: int64

In [50]:
successes = [conversions['control'], conversions['tratamiento']]
successes

[779, 820]

In [51]:
observations = [group_totals['control'], group_totals['tratamiento']]
observations

[4965, 5035]

In [52]:
z_stat, p_value = proportions_ztest(successes , observations)

print(f"Z statistic: {z_stat}")
print(f"P-value: {p_value}")

Z statistic: -0.8132782986429474
P-value: 0.41605851639119995


In [53]:
alpha = 0.05 

if p_value < alpha:
    print("Reject the null hypothesis: there is evidence of a difference.")
else:
    print("Fail to reject the null hypothesis: there is not enough evidence of a difference.")

Fail to reject the null hypothesis: there is not enough evidence of a difference.


### Interpretation of Results and A/B Test Conclusion

**Statistical result:**

- Z statistic: `-0.8133`
- P-value: `0.4161`
- Significance level ($\alpha$): `0.05`

**Analytical conclusion:** since the p-value (0.4161) is greater than the established significance level (alpha = 0.05), **there is not enough statistical evidence to reject the null hypothesis (H₀)**.

**Business recommendation:** the change to the checkout user interface (UI) does not produce a statistically significant impact on the purchase conversion rate. Therefore, it is not recommended to roll out the change to production based on this metric alone, since the observed increase is attributed to random sampling fluctuations.

## Step 6: Communicating the results (BI dashboard)

The results are communicated through an interactive dashboard built in **Tableau**, using the cleaned datasets exported in Step 1:

- `orders_clean.csv`
- `catalog_clean.csv`
- `marketing_clean.csv`

**Dashboard:** [View the interactive dashboard on Tableau Public](https://public.tableau.com/shared/R4CX4FGXN?:display_count=n&:origin=viz_share_link)